# Lab 2: regularización, validación y clasificación

Completa las funciones sin cambiar sus nombres ni firmas. El autograder
usará datos alternos y comprobará que entrenamiento, validación y prueba
conservan papeles distintos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

## Funciones requeridas

In [ ]:
def standardize_from_train(X_train, X_validation, X_test):
    # TODO: ajustar StandardScaler solo con X_train y transformar los tres conjuntos
    raise NotImplementedError


def sigmoid(z):
    # TODO: implementar sigmoide vectorizada y estable
    raise NotImplementedError


def log_loss_binary(y_true, proba, eps=1e-15):
    # TODO: implementar log-loss binaria con clipping
    raise NotImplementedError


def ridge_closed_form(X, y, alpha):
    # TODO: usar X.T@X + alpha*P, con P[0, 0] = 0
    raise NotImplementedError


def select_regularization_alpha(
    X_train,
    y_train,
    X_validation,
    y_validation,
    alphas,
    penalty="ridge",
):
    # TODO: comparar Ridge o Lasso usando únicamente el error de validación
    # Devuelve (mejor_alpha, {alpha: mse_validacion}).
    raise NotImplementedError


def classification_metrics(y_true, proba, threshold=0.5):
    # TODO: devolver accuracy, precision, recall y f1
    raise NotImplementedError


def bernoulli_standard_error(indicators):
    # TODO: error estándar plug-in de la media de indicadores 0/1
    raise NotImplementedError


def choose_threshold_for_recall(y_true, proba, min_recall=0.90):
    # TODO: devolver el umbral más alto que cumpla el recall mínimo
    raise NotImplementedError

## Protocolo de evaluación

Completa cada valor con una de las etiquetas permitidas en el enunciado.
El protocolo incluye ahora el modelo poblacional Bernoulli y el carácter
aleatorio de una métrica de validación.

In [ ]:
protocol_answers = {
    "fit_scaler_on": "",
    "select_regularization_on": "",
    "select_threshold_on": "",
    "test_set_role": "",
    "logistic_population_model": "",
    "validation_score_status": "",
}
probability_answers = protocol_answers

## Experimento de regresión

Construye particiones de entrenamiento, validación y prueba. Ajusta el
escalador en entrenamiento, compara Ridge y Lasso en validación y consulta
prueba una sola vez después de fijar el modelo y `alpha`.

In [ ]:
diabetes = load_diabetes()
X_train, X_temp, y_train, y_temp = train_test_split(
    diabetes.data, diabetes.target, test_size=0.40, random_state=2105
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=2105
)
X_train_s, X_validation_s, X_test_s = standardize_from_train(
    X_train, X_validation, X_test
)

alphas = [0.01, 0.1, 1.0, 10.0]
ridge_alpha, ridge_scores = select_regularization_alpha(
    X_train_s, y_train, X_validation_s, y_validation, alphas, penalty="ridge"
)
lasso_alpha, lasso_scores = select_regularization_alpha(
    X_train_s, y_train, X_validation_s, y_validation, alphas, penalty="lasso"
)

candidates = [
    ("ridge", ridge_alpha, ridge_scores[ridge_alpha]),
    ("lasso", lasso_alpha, lasso_scores[lasso_alpha]),
]
selected_penalty, selected_alpha, selected_validation_mse = min(
    candidates, key=lambda row: row[2]
)
if selected_penalty == "ridge":
    selected_model = Ridge(alpha=selected_alpha)
else:
    selected_model = Lasso(alpha=selected_alpha, max_iter=20000, tol=1e-8)
selected_model.fit(X_train_s, y_train)
test_prediction = selected_model.predict(X_test_s)

print("selección:", selected_penalty, selected_alpha)
print("MSE validación:", selected_validation_mse)
print("MSE prueba, consultado una vez:", mean_squared_error(y_test, test_prediction))

## Experimento de clasificación

Ajusta el modelo en entrenamiento, elige el umbral en validación y reporta
las métricas finales en prueba.

In [ ]:
cancer = load_breast_cancer()
X_train, X_temp, y_train, y_temp = train_test_split(
    cancer.data,
    cancer.target,
    test_size=0.40,
    random_state=2105,
    stratify=cancer.target,
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=2105,
    stratify=y_temp,
)
classifier = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=5000, random_state=2105)
)
classifier.fit(X_train, y_train)
validation_proba = classifier.predict_proba(X_validation)[:, 1]
selected_threshold = choose_threshold_for_recall(
    y_validation, validation_proba, min_recall=0.95
)

test_proba = classifier.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)
print("umbral elegido en validación:", selected_threshold)
print(
    "métricas finales de prueba:",
    classification_metrics(y_test, test_proba, threshold=selected_threshold),
)
print("SE plug-in de accuracy:", bernoulli_standard_error(test_pred == y_test))

## Interpretación manual

**Pregunta.** ¿Qué modelo o umbral recomendarías, qué cantidad poblacional intenta estimar la métrica y qué incertidumbre comunicarías antes de usarlo?

Responde en la celda siguiente, en 100-160 palabras. Cita resultados concretos del notebook y no incluyas tu nombre ni tu código estudiantil en la respuesta.

<!-- MATE2105:MANUAL_RESPONSE id=lab02_recomendacion -->
**Tu respuesta (100-160 palabras):**

[Escribe aquí tu respuesta. Cita resultados numéricos, identifica la cantidad poblacional y comunica una limitación o fuente de incertidumbre.]